In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from tabulate import tabulate
from fastridge import RidgeEM, RidgeLOOCV
from experiments import RealDataExperiment
from data import get_dataset

In [ ]:
datasets = ['yacht', 'diabetes']
targets  = ['Residuary_resistance', 'target']
names    = ['yacht', 'diabetes']

dataframes = [get_dataset(name) for name in datasets]

estimators = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results = RealDataExperiment(dataframes, targets, names, estimators, n_iterations=10, seed=1, verbose=False)

table = {}
for data_name, data_result in results.items():
    em_time   = data_result['EM']['time']
    cv_time   = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {est: data_result[est]['r2'] for est in data_result}
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']      = data_result['EM']['p']
    row['n_train'] = data_result['EM']['n_train']
    row['n:p']    = data_result['EM']['n_train'] / data_result['EM']['p']
    table[data_name] = row
tdf = pd.DataFrame(table)
print(tabulate(tdf.transpose().sort_values('n_train', ascending=False),
               headers='keys', tablefmt='psql', floatfmt='.2f'))

In [ ]:
# Full experiment matching legacy notebook — skip-execution in CI
datasets_full = ['abalone', 'automobile', 'autompg', 'airfoil',
                 'crime', 'concrete', 'diabetes',
                 'facebook', 'forest', 'parkinsons', 'real_estate',
                 'student', 'yacht']
targets_full = [
    'Rings', 'price', 'mpg', 'scaled-sound-pressure',
    'ViolentCrimesPerPop', 'Concrete Compressive Strength',
    'target', 'Total Interactions', 'area',
    'motor_UPDRS', 'Y house price of unit area', 'G3',
    'Residuary_resistance'
]
names_full = datasets_full

dataframes_full = [get_dataset(name) for name in datasets_full]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results_full = RealDataExperiment(dataframes_full, targets_full, names_full,
                                  estimators_full, n_iterations=100, seed=1, verbose=False)

table_full = {}
for data_name, data_result in results_full.items():
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {est: data_result[est]['r2'] for est in data_result}
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']      = data_result['EM']['p']
    row['n_train'] = data_result['EM']['n_train']
    row['n:p']    = data_result['EM']['n_train'] / data_result['EM']['p']
    table_full[data_name] = row
tdf_full = pd.DataFrame(table_full)
print(tabulate(tdf_full.transpose().sort_values('n_train', ascending=False),
               headers='keys', tablefmt='psql', floatfmt='.2f'))